# Dialogue summaries: a QLoRA evidence notebook
**Name: Priyanshi Goyal · Roll number: A24ARIU0009**

This new workflow separates resource estimates, answer-only data preparation, a paired before/after
experiment and a blind human review export. It uses Qwen2.5-1.5B-Instruct with NF4 double quantization
and LoRA on all seven decoder projection types. Automatic results are saved alongside individual
summaries; a separate rating sheet leaves human judgments genuinely unfilled.

Task provenance: [course QLoRA lab](https://github.com/bk-anupam/generative_ai_shared/tree/master/module3/labs/qlora).
Dataset: [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum), CC BY-NC-SA 4.0;
model: [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), Apache-2.0.
**Execution status: training, scores and human judgments are not yet measured.**


## Environment
Use a CUDA GPU with compatible `torch`, `transformers`, `accelerate`, `peft`, `bitsandbytes`,
`datasets`, `pandas` and `rouge_score` installed. Internet is needed for the initial model/data download.
The default training sample is 2,000 dialogues. Change this only before the experiment and record it.


In [ ]:
import os, json, time, random, gc, math
from pathlib import Path
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoTokenizer, Trainer, TrainingArguments
SEED = 42
transformers.set_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT_OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
print({'student': 'Priyanshi Goyal', 'roll': 'A24ARIU0009',
       'torch': torch.__version__, 'transformers': transformers.__version__, 'device': DEVICE})


In [ ]:
OUT=ROOT_OUT/'priyanshi_qlora_A24ARIU0009'
OUT.mkdir(parents=True,exist_ok=True)
CHECKPOINT='Qwen/Qwen2.5-1.5B-Instruct'
N_TRAIN,N_EVAL,MAX_LENGTH=2000,60,512
TOTAL,LINEAR,OTHER,ADAPTER=1543714304,1310195712,233518592,18464768
# Idealized model/optimizer storage only; activations and allocator overhead are excluded.
estimates={'full_ft':16*TOTAL/1e9,
           'lora':(2*TOTAL+16*ADAPTER)/1e9,
           'qlora':(((4+.127)/8)*LINEAR+4*OTHER+16*ADAPTER)/1e9}
print(pd.Series(estimates,name='estimated_GB_before_activations'))
assert abs(estimates['qlora']-1.91)<.01
assert DEVICE=='cuda', 'Quantized training requires a supported CUDA GPU for this notebook'
from datasets import load_dataset
raw=load_dataset('knkarthick/dialogsum')
tokenizer=AutoTokenizer.from_pretrained(CHECKPOINT)
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
train_rows=raw['train'].shuffle(seed=SEED).select(range(min(N_TRAIN,len(raw['train']))))
eval_rows=raw['validation'].shuffle(seed=SEED).select(range(min(N_EVAL,len(raw['validation']))))
# The test split is never accessed.


In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from rouge_score import rouge_scorer

def pack_answer(prefix, answer, stop_id):
    if not prefix or not answer: raise ValueError('Empty prompt or answer')
    target=answer+[stop_id]
    ids=prefix+target
    return {'input_ids':ids,'attention_mask':[1]*len(ids),'labels':[-100]*len(prefix)+target}

def collate_answers(batch):
    width=max(len(row['input_ids']) for row in batch)
    output={key:torch.full((len(batch),width),fill,dtype=torch.long)
            for key,fill in [('input_ids',tokenizer.pad_token_id),('attention_mask',0),('labels',-100)]}
    for i,row in enumerate(batch):
        n=len(row['input_ids'])
        for key in output: output[key][i,:n]=torch.tensor(row[key],dtype=torch.long)
    return output

@torch.inference_mode()
def decode_prompts(model, prompts, max_new):
    model.eval()
    predictions=[]
    for prompt in prompts:
        ids=torch.tensor([prompt],dtype=torch.long,device=model.device)
        result=model.generate(input_ids=ids,attention_mask=torch.ones_like(ids),
            do_sample=False,max_new_tokens=max_new,pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id)
        predictions.append(tokenizer.decode(result[0,len(prompt):],skip_special_tokens=True).strip())
    return predictions

def rouge_table(predictions, references):
    assert len(predictions)==len(references)>0
    scorer=rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'],use_stemmer=True)
    pairs=[scorer.score(reference,prediction) for prediction,reference in zip(predictions,references)]
    return {key:100*np.mean([row[key].fmeasure for row in pairs]) for key in ['rouge1','rouge2','rougeL']}

def fit_model(model, args, train_data, validation=None):
    trainer=Trainer(model=model,args=args,train_dataset=train_data,eval_dataset=validation,
                    data_collator=collate_answers)
    if DEVICE=='cuda': torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
    started=time.perf_counter()
    trainer.train()
    if DEVICE=='cuda': torch.cuda.synchronize()
    stats={'train_seconds':time.perf_counter()-started,
           'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
           'peak_gpu_gb':torch.cuda.max_memory_allocated()/1e9 if DEVICE=='cuda' else None}
    if validation is not None: stats['validation_loss']=trainer.evaluate()['eval_loss']
    Path(args.output_dir).mkdir(parents=True,exist_ok=True)
    Path(args.output_dir,'training_log.json').write_text(json.dumps(trainer.state.log_history,indent=2))
    from accelerate.utils import extract_model_from_parallel
    extract_model_from_parallel(model, keep_fp32_wrapper=False)
    return trainer,stats

# A padding check explicitly preserves real EOS supervision even when PAD == EOS.
probe=collate_answers([pack_answer([3,4],[5],tokenizer.eos_token_id),pack_answer([3],[6],tokenizer.eos_token_id)])
assert probe['labels'][0,-1].item()==tokenizer.eos_token_id
assert probe['labels'][1,-1].item()==-100 and probe['attention_mask'][1,-1].item()==0
assert (probe['labels'][:,0]==-100).all()


In [ ]:
INSTRUCTION='Summarize this conversation in one or two sentences. Preserve speaker labels and do not invent facts.'
STOP=tokenizer.convert_tokens_to_ids('<|im_end|>')
assert STOP==tokenizer.eos_token_id

def dialogue_prompt(dialogue):
    return tokenizer.apply_chat_template([{'role':'user','content':INSTRUCTION+'\n\n'+dialogue}],
                                         tokenize=True,add_generation_prompt=True)

def dialogue_example(row):
    return pack_answer(dialogue_prompt(row['dialogue']),tokenizer.encode(row['summary'],add_special_tokens=False),STOP)

all_examples=[dialogue_example(row) for row in train_rows]
train_data=[row for row in all_examples if len(row['input_ids'])<=MAX_LENGTH]
assert train_data, 'No complete examples fit the token budget'
print({'retained':len(train_data),'dropped_long':len(all_examples)-len(train_data),
       'scored_tokens':sum(sum(x!=-100 for x in row['labels']) for row in train_data)})
prompts=[dialogue_prompt(row['dialogue']) for row in eval_rows]
references=list(eval_rows['summary'])
(OUT/'run_config.json').write_text(json.dumps({'seed':SEED,'checkpoint':CHECKPOINT,
    'requested_train':N_TRAIN,'retained_train':len(train_data),'eval_count':len(eval_rows),
    'max_length':MAX_LENGTH,'instruction':INSTRUCTION,'validation_ids':list(eval_rows['id'])},indent=2))


## Quantized reference, then train only adapters
The baseline uses the same quantized model and deterministic decoder as the tuned run. This controls
quantization and prompt differences. Double quantization stores quantization constants compactly;
NF4 stores linear weights with a distribution-aware 4-bit code. LoRA supplies trainable low-rank
updates while the base remains frozen. Preparation can upcast unquantized weights, so actual storage
is not simply one half-byte per parameter.


In [ ]:
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import bitsandbytes as bnb
quantization=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(CHECKPOINT,quantization_config=quantization,
    device_map={'':0},dtype=torch.float16)
assert sum(isinstance(layer,bnb.nn.Linear4bit) for layer in model.modules())>0
base_predictions=decode_prompts(model,prompts,96)
model=prepare_model_for_kbit_training(model,gradient_checkpointing_kwargs={'use_reentrant':False})
model=get_peft_model(model,LoraConfig(task_type='CAUSAL_LM',r=16,lora_alpha=32,lora_dropout=.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],bias='none'))
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
assert trainable==ADAPTER
assert all('lora_' in name for name,p in model.named_parameters() if p.requires_grad)
model.config.use_cache=False
args=TrainingArguments(output_dir=str(OUT/'training'),num_train_epochs=1,
    per_device_train_batch_size=2,gradient_accumulation_steps=8,learning_rate=2e-4,
    optim='paged_adamw_8bit',fp16=True,bf16=False,gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant':False},logging_steps=10,
    save_strategy='no',report_to='none',seed=SEED,remove_unused_columns=False)
trainer,training_stats=fit_model(model,args,train_data)
assert all(math.isfinite(row['loss']) for row in trainer.state.log_history if 'loss' in row)
model.gradient_checkpointing_disable()
model.config.use_cache=True
tuned_predictions=decode_prompts(model,prompts,96)
model.save_pretrained(OUT/'adapter')
tokenizer.save_pretrained(OUT/'adapter')
(OUT/'training_stats.json').write_text(json.dumps(training_stats,indent=2))
print(training_stats)


In [ ]:
scorer=rouge_scorer.RougeScorer(['rougeL'],use_stemmer=True)
base_l=np.array([scorer.score(r,p)['rougeL'].fmeasure for r,p in zip(references,base_predictions)])
tuned_l=np.array([scorer.score(r,p)['rougeL'].fmeasure for r,p in zip(references,tuned_predictions)])
delta=tuned_l-base_l
rng=np.random.default_rng(SEED)
boot=np.array([rng.choice(delta,size=len(delta),replace=True).mean()*100 for _ in range(2000)])
interval=np.quantile(boot,[.025,.975])
metrics=pd.DataFrame([{'model':name,**rouge_table(predictions,references),
    'mean_words':np.mean([len(p.split()) for p in predictions])}
    for name,predictions in [('quantized_base',base_predictions),('qlora',tuned_predictions)]])
metrics.to_csv(OUT/'metrics.csv',index=False)
paired=pd.DataFrame({'id':list(eval_rows['id']),'dialogue':list(eval_rows['dialogue']),
    'reference':references,'base':base_predictions,'qlora':tuned_predictions,'rougeL_delta':delta*100})
paired.to_csv(OUT/'paired_predictions.csv',index=False)
print(metrics.to_string(index=False))
print({'rougeL_mean_change':delta.mean()*100,'paired_bootstrap_95_percent':interval.tolist(),
       'wins':int((delta>0).sum()),'ties':int((delta==0).sum())})


## Blind review and uncertainty
ROUGE measures reference word overlap rather than factual correctness. The paired bootstrap estimates
sampling uncertainty across these validation examples, not training-seed uncertainty or generalization
to other domains. Have a reviewer score A and B for faithfulness and coverage (0 = poor, 1 = partial,
2 = good), and add a note about unsupported facts. Keep the answer key hidden until rating is complete.
The full dialogue is included to avoid judging against a truncated conversation.


In [ ]:
review_path=OUT/'blind_review.csv'
key_path=OUT/'blind_key.json'
# Preserve existing human work on reruns. Use a new output directory for a new experiment.
if not review_path.exists() and not key_path.exists():
    order=random.Random(SEED)
    review,key=[],{}
    for i in order.sample(range(len(paired)),min(8,len(paired))):
        first=order.choice(['base','qlora']); second='qlora' if first=='base' else 'base'
        row=paired.iloc[i]
        review.append({'id':row.id,'dialogue':row.dialogue,'reference':row.reference,
            'A':row[first],'B':row[second],'A_faithfulness':None,'B_faithfulness':None,
            'A_coverage':None,'B_coverage':None,'note':''})
        key[str(row.id)]={'A':first,'B':second}
    pd.DataFrame(review).to_csv(review_path,index=False)
    key_path.write_text(json.dumps(key,indent=2))
else:
    print('Existing review artifacts preserved. Use a new OUT for a new run.')
print('Reviewer sheet:',review_path)


In [ ]:
# Re-run only after the reviewer saves their scores. Missing scores are never treated as zeros.
ratings=pd.read_csv(review_path)
columns=[f'{side}_{criterion}' for side in ['A','B'] for criterion in ['faithfulness','coverage']]
if ratings[columns].isna().any().any():
    print('Human evaluation pending; no human result reported.')
else:
    assert ratings[columns].isin([0,1,2]).all().all()
    key=json.loads(key_path.read_text())
    long=[]
    for row in ratings.to_dict('records'):
        for side in ['A','B']:
            long.append({'model':key[str(row['id'])][side],
                'faithfulness':row[f'{side}_faithfulness'],'coverage':row[f'{side}_coverage']})
    summary=pd.DataFrame(long).groupby('model').mean()
    summary.to_csv(OUT/'human_results.csv')
    print(summary)


## Conclusions to write from the run
Compare automatic scores with blind judgments and cite specific disagreements. Contrast measured
peak GPU memory with the idealized estimate: activations, temporary buffers, upcasting and optimizer
implementation explain differences. A decreasing loss is useful evidence, but a small final increase
does not automatically mean failure. Report failed runs and the exact environment.

The saved adapter requires the original checkpoint and matching quantization configuration to reload;
it is not a standalone full model. Do not merge into 4-bit weights as a substitute for evaluating a
properly reloaded adapter. The memory estimate is a planning calculation, not a measured guarantee.

References: [QLoRA](https://arxiv.org/abs/2305.14314), [LoRA](https://arxiv.org/abs/2106.09685),
and the dataset, model and course sources above.
